# Drug Safety Signal Summary — SQL

SQL-cell equivalent of `drug_safety_signal_summary.ipynb`, using `%%sql`
cells instead of the DataFrame API. Same result: reads
`pharmacovigilance.adverse_events` and persists
`pharmacovigilance.drug_safety_signal_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/fda_adverse_event_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: safety signal by drug/reaction

In [ ]:
%%sql
CREATE OR REPLACE TABLE pharmacovigilance.drug_safety_signal_summary_v1 AS
SELECT
    drug_name,
    reaction_term,
    COUNT(*) AS report_count,
    SUM(CASE WHEN serious = '1' THEN 1 ELSE 0 END) AS serious_count,
    SUM(CASE WHEN seriousness_death = '1' THEN 1 ELSE 0 END) AS death_count,
    ROUND(SUM(CASE WHEN serious = '1' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS serious_rate_pct
FROM pharmacovigilance.adverse_events
GROUP BY drug_name, reaction_term
ORDER BY report_count DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_signals FROM pharmacovigilance.drug_safety_signal_summary_v1

In [ ]:
%%sql
SELECT * FROM pharmacovigilance.drug_safety_signal_summary_v1
ORDER BY report_count DESC
LIMIT 10